# Week 3 Exercise – Synthetic Data Generator

Build a tool that generates synthetic datasets. Describe the kind of data you need, how you want it structured, its purpose, and generate many records. Use a variety of models (including open-source). Try different sizes; optionally try quantized vs non-quantized. Add a Gradio UI.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

OLLAMA_BASE = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE, api_key="ollama")

openrouter_key = os.getenv("OPENAI_API_KEY")
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1") if openrouter_key else None

MODELS_OLLAMA = ["llama3.2", "qwen2.5:7b", "mistral"]
MODEL_API = "openai/gpt-4o-mini"
MODELS = MODELS_OLLAMA + ([MODEL_API] if openrouter else [])

In [ ]:
SYSTEM_PROMPT = """You are a synthetic data generator. From the user's description you produce datasets: realistic, structured, and suitable for the stated purpose. Output only the requested data in the requested format. No preamble or explanation. Use plausible values; no real personal data."""

def build_prompt(description: str, structure: str, purpose: str, num_records: int, out_format: str) -> str:
    return f"""Description: {description}
Structure: {structure}
Purpose: {purpose}
Generate exactly {num_records} records. Output format: {out_format}. Output only the data, no other text."""

In [ ]:
def generate(description: str, structure: str, purpose: str, num_records: int, out_format: str, model_name: str) -> str:
    description = description or ""
    if not description.strip():
        return ""
    n = max(1, min(500, int(num_records))) if isinstance(num_records, (int, float)) else 10
    prompt = build_prompt(description, structure, purpose, n, out_format)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    if model_name == MODEL_API and openrouter:
        r = openrouter.chat.completions.create(model=MODEL_API, messages=messages, max_tokens=4096)
    else:
        r = ollama.chat.completions.create(model=model_name, messages=messages, max_tokens=4096)
    return (r.choices[0].message.content or "").strip()

In [ ]:
def run_ui(description, structure, purpose, num_records, out_format, model_name):
    return generate(description or "", structure or "", purpose or "", num_records or 10, out_format or "JSON", model_name or MODELS[0])

In [ ]:
with gr.Blocks(title="Synthetic Data Generator") as app:
    gr.Markdown("Describe the data you need, its structure and purpose. Choose format, number of records, and model.")
    with gr.Row():
        description = gr.Textbox(label="Data description", placeholder="e.g. employee records for a tech company", lines=2)
    with gr.Row():
        structure = gr.Textbox(label="Structure / schema", placeholder="e.g. id, name, role, department, hire_date", lines=2)
        purpose = gr.Textbox(label="Purpose", placeholder="e.g. training a classifier", lines=2)
    with gr.Row():
        num_records = gr.Number(label="Number of records", value=10, minimum=1, maximum=500)
        out_format = gr.Dropdown(choices=["JSON", "CSV", "JSONL"], value="JSON", label="Output format")
        model_name = gr.Dropdown(choices=MODELS, value=MODELS[0], label="Model")
    btn = gr.Button("Generate")
    output = gr.Textbox(label="Generated data", lines=16)
    btn.click(fn=run_ui, inputs=[description, structure, purpose, num_records, out_format, model_name], outputs=output)
app.launch()

Optional: HuggingFace model with 4-bit quantization (needs GPU, HF token, `transformers`, `bitsandbytes`). Run in Colab or locally if you have the dependencies.

In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    HF_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    qconfig = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
    tokenizer.pad_token = tokenizer.eos_token
    model_q = AutoModelForCausalLM.from_pretrained(HF_MODEL, device_map="auto", quantization_config=qconfig)
    def generate_hf_quant(description, structure, purpose, num_records, out_format):
        prompt = build_prompt(description, structure, purpose, max(1, min(50, int(num_records))), out_format)
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model_q.device)
        out = model_q.generate(**inputs, max_new_tokens=1024, do_sample=True, temperature=0.7)
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print("HF quantized model ready. Call generate_hf_quant(description, structure, purpose, num_records, out_format).")
except Exception as e:
    print("Skip HF quantization:", e)